In [12]:
# ============================================================
# USER-EDITABLE CO WHITE DWARF SETTINGS
# ============================================================

# ---------- STAR ----------
MASS_MSUN = 1.5
INITIAL_Z = 0.0142

# ---------- SPEED / TERMINATION ----------
# Higher = potentially faster but less detailed evolution
MAX_MODEL_NUMBER = 30000

# Stop removing the envelope when log10(Teff) reaches this value
LOG_TEFF_LIMIT = 3.8

# ---------- MESA NUMERICAL SETTINGS ----------
# Larger values generally mean fewer/simpler mesh points
MESH_DELTA_COEFF = 0.5



# More stringent timestep control
VARCONTROL_TARGET = 0.0001

# Allow MESA to take extremely small timesteps during helium ignition
MIN_TIMESTEP_LIMIT = 1e-8

# Maximum timestep in years
MAX_YEARS_FOR_TIMESTEP = 1e6


# ---------- OUTPUT ----------
HISTORY_INTERVAL = 100
PROFILE_INTERVAL = 100
TERMINAL_INTERVAL = 100

# ---------- RUN CONTROL ----------
OVERWRITE_RUN = True
RUN_MESA = True


print("======================================")
print("CO-WD RUN SETTINGS")
print("======================================")
print(f"Initial mass:        {MASS_MSUN} Msun")
print(f"Initial metallicity: Z = {INITIAL_Z}")
print(f"Max model number:    {MAX_MODEL_NUMBER}")
print(f"log(Teff) limit:     {LOG_TEFF_LIMIT}")
print(f"Mesh coefficient:    {MESH_DELTA_COEFF}")
print(f"Varcontrol target:   {VARCONTROL_TARGET}")
print(f"Max timestep:        {MAX_YEARS_FOR_TIMESTEP} yr")
print(f"History interval:    {HISTORY_INTERVAL}")
print(f"Profile interval:    {PROFILE_INTERVAL}")

# ============================================================
# PATHS
# ============================================================

import os
import re
import shutil
import subprocess
from pathlib import Path

os.environ["MESA_DIR"] = "/Users/mchau/mesa"

PROJECT_ROOT = Path("/Users/mchau/mesa_tutorial")
WD_DIR = PROJECT_ROOT / "WD" / "make_co_wd"

print("MESA_DIR:", os.environ["MESA_DIR"])
print("Source directory:", WD_DIR)

CO-WD RUN SETTINGS
Initial mass:        1.5 Msun
Initial metallicity: Z = 0.0142
Max model number:    30000
log(Teff) limit:     3.8
Mesh coefficient:    0.5
Varcontrol target:   0.0001
Max timestep:        1000000.0 yr
History interval:    100
Profile interval:    100
MESA_DIR: /Users/mchau/mesa
Source directory: /Users/mchau/mesa_tutorial/WD/make_co_wd


In [13]:
# ============================================================
# CREATE CUSTOM RUN DIRECTORY
# ============================================================

mass_label = f"{MASS_MSUN:.1f}".replace(".", "p")
z_label = f"{INITIAL_Z:.4f}".replace(".", "p")

RUN_DIR = (
    PROJECT_ROOT
    / "WD"
    / f"CO_WD_{mass_label}Msun_Z{z_label}"
)

print("RUN_DIR =", RUN_DIR)

if RUN_DIR.exists():

    if OVERWRITE_RUN:
        print("Removing previous run...")
        shutil.rmtree(RUN_DIR)
    else:
        raise FileExistsError(
            f"{RUN_DIR} already exists. "
            "Set OVERWRITE_RUN = True or choose different parameters."
        )

RUN_DIR.mkdir(parents=True)

print("Created:", RUN_DIR)


# ============================================================
# COPY MESA SETUP
# ============================================================

# IMPORTANT:
# Use the newly rebuilt MESA executable (HDF5 310).
# Do NOT copy "star" from WD_DIR because that executable
# is linked against the old HDF5 311 libraries.

STAR_SOURCE = Path("/Users/mchau/mesa/star/test/star")
STAR_DEST = RUN_DIR / "star"


# ============================================================
# COPY CORRECT MESA EXECUTABLE
# ============================================================

if not STAR_SOURCE.exists():
    raise FileNotFoundError(
        f"New MESA star executable not found:\n{STAR_SOURCE}"
    )

shutil.copy2(STAR_SOURCE, STAR_DEST)
STAR_DEST.chmod(0o755)

print("\nCopied MESA executable:")
print("  From:", STAR_SOURCE)
print("  To:  ", STAR_DEST)


# ============================================================
# VERIFY HDF5 VERSION
# ============================================================

check = subprocess.run(
    ["otool", "-L", str(STAR_DEST)],
    capture_output=True,
    text=True
)

print("\n===== STAR HDF5 LIBRARIES =====")

for line in check.stdout.splitlines():
    if "hdf5" in line.lower():
        print(line)

if "libhdf5_fortran.310.dylib" not in check.stdout:
    raise RuntimeError(
        "\nWRONG MESA EXECUTABLE!\n"
        "Expected libhdf5_fortran.310.dylib, "
        "but the copied star executable is not linked to HDF5 310."
    )

print("\n✓ Correct MESA executable confirmed: HDF5 310")


# ============================================================
# COPY REMAINING MESA SETUP FILES
# ============================================================

files_to_copy = [
    "inlist",
    "inlist_common",
    "inlist_remove_env",
    "inlist_remove_env_header",
    "inlist_settle",
    "inlist_settle_header",
    "inlist_zams",
    "inlist_zams_header",
    "inlist_co_core",
    "inlist_co_core_header",
    "inlist_to_end_he_core_burn",
    "inlist_to_end_he_core_burn_header",
    "history_columns.list",
    "profile_columns.list",
    "testhub.yml",
    "rn",
    "rn1",
    "re",
    "ck",
    "zams.mod",
    "end_he_core_burn.mod",
    "co_core.mod",
]

print("\n===== COPYING MESA SETUP FILES =====")

for name in files_to_copy:

    source = WD_DIR / name
    destination = RUN_DIR / name

    if not source.exists():
        print("WARNING — missing:", name)
        continue

    if source.is_file():
        shutil.copy2(source, destination)
        print("Copied:", name)


# ============================================================
# DONE
# ============================================================

print("\n============================================================")
print("MESA SETUP COPIED SUCCESSFULLY")
print("============================================================")
print("Run directory:", RUN_DIR)

RUN_DIR = /Users/mchau/mesa_tutorial/WD/CO_WD_1p5Msun_Z0p0142
Removing previous run...
Created: /Users/mchau/mesa_tutorial/WD/CO_WD_1p5Msun_Z0p0142

Copied MESA executable:
  From: /Users/mchau/mesa/star/test/star
  To:   /Users/mchau/mesa_tutorial/WD/CO_WD_1p5Msun_Z0p0142/star

===== STAR HDF5 LIBRARIES =====
	@rpath/lib/libhdf5_fortran.310.dylib (compatibility version 314.0.0, current version 314.2.0)
	@rpath/lib/libhdf5.310.dylib (compatibility version 316.0.0, current version 316.1.0)

✓ Correct MESA executable confirmed: HDF5 310

===== COPYING MESA SETUP FILES =====
Copied: inlist
Copied: inlist_common
Copied: inlist_remove_env
Copied: inlist_remove_env_header
Copied: inlist_settle
Copied: inlist_settle_header
Copied: inlist_zams
Copied: inlist_zams_header
Copied: inlist_co_core
Copied: inlist_co_core_header
Copied: inlist_to_end_he_core_burn
Copied: inlist_to_end_he_core_burn_header
Copied: history_columns.list
Copied: profile_columns.list
Copied: testhub.yml
Copied: rn
Copied: 

In [17]:
# ============================================================
# APPLY EDITABLE SETTINGS TO MESA INLISTS
# ============================================================

# ------------------------------------------------------------
# inlist_common
# ------------------------------------------------------------

common = RUN_DIR / "inlist_common"
text = common.read_text()

text, n_mass = re.subn(
    r"(^\s*initial_mass\s*=\s*)[-+0-9.eEdD]+",
    rf"\g<1>{MASS_MSUN:.6g}d0",
    text,
    flags=re.MULTILINE
)

text, n_z = re.subn(
    r"(^\s*initial_z\s*=\s*)[-+0-9.eEdD]+",
    rf"\g<1>{INITIAL_Z:.6g}d0",
    text,
    flags=re.MULTILINE
)

common.write_text(text)


# ------------------------------------------------------------
# inlist_remove_env
# ------------------------------------------------------------

remove_env = RUN_DIR / "inlist_remove_env"
text = remove_env.read_text()

text, n_max = re.subn(
    r"(^\s*max_model_number\s*=\s*)\d+",
    rf"\g<1>{MAX_MODEL_NUMBER}",
    text,
    flags=re.MULTILINE
)

text, n_teff = re.subn(
    r"(^\s*log_Teff_upper_limit\s*=\s*)[-+0-9.eEdD]+",
    rf"\g<1>{LOG_TEFF_LIMIT}",
    text,
    flags=re.MULTILINE
)

text, n_min_dt = re.subn(
    r"(^\s*min_timestep_limit\s*=\s*)[-+0-9.eEdD]+",
    rf"\g<1>1d-8",
    text,
    flags=re.MULTILINE
)

remove_env.write_text(text)


# ------------------------------------------------------------
# Check replacements
# ------------------------------------------------------------

print("Settings updated:")
print("  initial_mass:", n_mass)
print("  initial_z:", n_z)
print("  max_model_number:", n_max)
print("  log_Teff_upper_limit:", n_teff)

# ============================================================
# APPLY SPEED / OUTPUT SETTINGS
# ============================================================

common = RUN_DIR / "inlist_common"
text = common.read_text()

# Mesh
text = re.sub(
    r"(^\s*mesh_delta_coeff\s*=\s*)[-+0-9.eEdD]+",
    rf"\g<1>{MESH_DELTA_COEFF}",
    text,
    flags=re.MULTILINE
)

# Timestep control
text = re.sub(
    r"(^\s*varcontrol_target\s*=\s*)[-+0-9.eEdD]+",
    rf"\g<1>{VARCONTROL_TARGET}",
    text,
    flags=re.MULTILINE
)

text = re.sub(
    r"(^\s*max_years_for_timestep\s*=\s*)[-+0-9.eEdD]+",
    rf"\g<1>{MAX_YEARS_FOR_TIMESTEP:.6g}",
    text,
    flags=re.MULTILINE
)

# Output frequency
text = re.sub(
    r"(^\s*history_interval\s*=\s*)\d+",
    rf"\g<1>{HISTORY_INTERVAL}",
    text,
    flags=re.MULTILINE
)

text = re.sub(
    r"(^\s*profile_interval\s*=\s*)\d+",
    rf"\g<1>{PROFILE_INTERVAL}",
    text,
    flags=re.MULTILINE
)

text = re.sub(
    r"(^\s*terminal_interval\s*=\s*)\d+",
    rf"\g<1>{TERMINAL_INTERVAL}",
    text,
    flags=re.MULTILINE
)

common.write_text(text)

print("Speed/output settings applied.")

Settings updated:
  initial_mass: 1
  initial_z: 1
  max_model_number: 1
  log_Teff_upper_limit: 1
Speed/output settings applied.


In [18]:
# ============================================================
# VERIFY FINAL SETTINGS
# ============================================================

print("\n========== inlist_common ==========")

text = (RUN_DIR / "inlist_common").read_text()

for line in text.splitlines():
    if any(key in line for key in [
        "initial_mass",
        "initial_z",
        "mesh_delta_coeff",
        "varcontrol_target",
        "max_years_for_timestep",
        "history_interval",
        "profile_interval",
        "terminal_interval",
    ]):
        print(line)


print("\n========== inlist_remove_env ==========")

text = (RUN_DIR / "inlist_remove_env").read_text()

for line in text.splitlines():
    if any(key in line for key in [
        "max_model_number",
        "log_Teff_upper_limit",
    ]):
        print(line)


========== inlist_common ==========
      initial_mass = 1.5d0 ! this default produces a ~0.6 Msun WD.
      ! initial_mass in the range 1.0-6.5 Msun will produce CO WDs
      initial_z = 0.0142d0
      mesh_delta_coeff = 0.5
      max_years_for_timestep = 1e+06
      varcontrol_target = 0.0001
      history_interval = 100
      terminal_interval = 100 

========== inlist_remove_env ==========
      required_termination_code_string = 'log_Teff_upper_limit'
      ! limit max_model_number as part of test_suite
      max_model_number = 30000
      log_Teff_upper_limit = 3.8


In [19]:
# ============================================================
# REMOVE OLD MESA MODELS
# ============================================================

old_models = [
    "zams.mod",
    "end_he_core_burn.mod",
    "co_core.mod",
    "co_wd.mod",
    "final.mod",
]

print("Removing old model files...")

for filename in old_models:
    path = RUN_DIR / filename

    if path.exists():
        path.unlink()
        print("  deleted:", filename)
    else:
        print("  not found:", filename)

print("Done.")

Removing old model files...
  deleted: zams.mod
  deleted: end_he_core_burn.mod
  deleted: co_core.mod
  not found: co_wd.mod
  not found: final.mod
Done.


In [ ]:
# ============================================================
# RUN MESA QUIETLY
# Full MESA output -> mesa_run.log
# Notebook only shows important information
# ============================================================

import subprocess
from pathlib import Path
import time

print("=" * 60)
print("STARTING MESA")
print("=" * 60)
print("Run directory:", RUN_DIR)
print("Mass:", MASS_MSUN, "Msun")
print("Z:", INITIAL_Z)
print()

log_file = RUN_DIR / "mesa_run.log"

start_time = time.time()

with open(log_file, "w") as log:
    result = subprocess.run(
        ["./rn"],
        cwd=RUN_DIR,
        stdout=log,
        stderr=subprocess.STDOUT,
        text=True
    )

elapsed = time.time() - start_time

print("=" * 60)
print("MESA FINISHED")
print("=" * 60)
print(f"Return code: {result.returncode}")
print(f"Runtime: {elapsed/60:.1f} minutes")
print(f"Full output saved to: {log_file}")
print()

# ------------------------------------------------------------
# Check important output files
# ------------------------------------------------------------

print("===== MODEL FILES =====")

models = [
    "zams.mod",
    "end_he_core_burn.mod",
    "co_core.mod",
    "co_wd.mod",
    "final.mod",
]

for model in models:
    path = RUN_DIR / model
    if path.exists():
        print(f"✓ {model}")
    else:
        print(f"✗ {model}")

print()

# ------------------------------------------------------------
# Check history
# ------------------------------------------------------------

history = RUN_DIR / "LOGS" / "history.data"

print("===== HISTORY =====")

if history.exists():
    print(f"✓ history.data created ({history.stat().st_size / 1024:.1f} KB)")
else:
    print("✗ history.data not found")

print()

# ------------------------------------------------------------
# Show only errors/warnings from the MESA log
# ------------------------------------------------------------

print("===== WARNINGS / ERRORS =====")

if log_file.exists():
    lines = log_file.read_text(errors="replace").splitlines()

    important = [
        line for line in lines
        if (
            "ERROR" in line.upper()
            or "WARNING" in line.upper()
            or "FAILED" in line.upper()
            or "termination code" in line.lower()
        )
    ]

    if important:
        for line in important[-20:]:
            print(line)
    else:
        print("No warnings/errors detected.")

print()
print("=" * 60)

STARTING MESA
Run directory: /Users/mchau/mesa_tutorial/WD/CO_WD_1p5Msun_Z0p0142
Mass: 1.5 Msun
Z: 0.0142



In [7]:
star_path = RUN_DIR / "star"

print("===== STAR BEING USED =====")
print(star_path)

check = subprocess.run(
    ["otool", "-L", str(star_path)],
    capture_output=True,
    text=True
)

print(check.stdout)

assert "libhdf5_fortran.310.dylib" in check.stdout, \
    "STOP: Jupyter is using the wrong HDF5/STAR executable!"

===== STAR BEING USED =====
/Users/mchau/mesa_tutorial/WD/CO_WD_1p5Msun_Z0p0142/star
/Users/mchau/mesa_tutorial/WD/CO_WD_1p5Msun_Z0p0142/star:
	@rpath/lib/libcrmath.dylib (compatibility version 0.0.0, current version 0.0.0)
	@rpath/lib/libcrlibm.0.dylib (compatibility version 1.0.0, current version 1.0.0)
	@rpath/math-slots/crmath/lib/liblapack95.dylib (compatibility version 0.0.0, current version 0.0.0)
	@rpath/math-slots/crmath/lib/libtmg.dylib (compatibility version 0.0.0, current version 0.0.0)
	@rpath/math-slots/crmath/lib/liblapack.dylib (compatibility version 0.0.0, current version 0.0.0)
	@rpath/math-slots/crmath/lib/libblas.dylib (compatibility version 0.0.0, current version 0.0.0)
	@rpath/lib/libhdf5_fortran.310.dylib (compatibility version 314.0.0, current version 314.2.0)
	@rpath/lib/libhdf5.310.dylib (compatibility version 316.0.0, current version 316.1.0)
	/usr/lib/libz.1.dylib (compatibility version 1.0.0, current version 1.2.11)
	@rpath/lib/libpgplot.dylib (compatibilit

In [6]:
print("RUN_DIR:", RUN_DIR)

print("\n===== CURRENT MODEL FILES =====")
for f in RUN_DIR.glob("*.mod"):
    print(f.name)

print("\n===== CURRENT INITIAL SETTINGS =====")
common = RUN_DIR / "inlist_common"

for line in common.read_text().splitlines():
    if "initial_mass" in line or "initial_z" in line:
        print(line)

RUN_DIR: /Users/mchau/mesa_tutorial/WD/CO_WD_1p5Msun_Z0p0142

===== CURRENT MODEL FILES =====

===== CURRENT INITIAL SETTINGS =====
      initial_mass = 1.5d0 ! this default produces a ~0.6 Msun WD.
      ! initial_mass in the range 1.0-6.5 Msun will produce CO WDs
      initial_z = 0.0142d0


In [21]:
for name in ["inlist_common", "inlist_remove_env"]:
    path = RUN_DIR / name
    print(f"\n--- {name} ---")

    for line in path.read_text().splitlines():
        if any(x in line for x in [
            "mesh_delta_coeff",
            "varcontrol_target",
            "min_timestep_limit",
            "max_years_for_timestep"
        ]):
            print(line.strip())


--- inlist_common ---
mesh_delta_coeff = 0.5
max_years_for_timestep = 1e+06
varcontrol_target = 0.0001

--- inlist_remove_env ---
min_timestep_limit = 1d-8 ! (seconds)
